# Audio Classification

> Labelling sound - tagging, keyword spotting, emotion, zero-shot: how the models work, the mid-2026 landscape, evaluation, and runnable transformers code on GPU or CPU.

- skip_showdoc: true
- skip_exec: true

## 1. What is Audio Classification?

Audio classification maps an audio clip to one or more **labels**. Common variants:

- **Sound event tagging** - what sounds are present (AudioSet's 527 classes, ESC-50).
- **Keyword spotting (KWS)** - detect command words ("yes", "stop").
- **Speech emotion recognition** - happy / sad / angry / neutral.
- **Speaker identification** and **language ID**.
- **Music genre / auto-tagging**, **acoustic scene classification**.

**Output.** A single label (multi-class) or many labels with scores (multi-label tagging).

**Neighbouring tasks:** Voice Activity Detection (a binary special case), ASR, and audio captioning.

---

## 2. Real-World Use Cases

Audio classification is the workhorse of "listen for something" systems. Most
deployments are **always-on**, which is why the binding constraints are power,
false-alarm rate and privacy far more often than top-1 accuracy.

| Use case | Domain | Consumes / produces | Dominant constraint |
|----------|--------|---------------------|---------------------|
| Wake words and keyword spotting | Consumer devices | Ring buffer of mic audio -> "is this the wake word?" | Milliwatts, tiny memory, and an extremely low false-accept rate |
| Music tagging and catalogue search | Streaming, media | Track -> genre, mood, instrument tags | Throughput and cost over millions of tracks; multi-label output |
| Safety and security event detection | Smart home, public safety (glass break, smoke alarm, baby cry) | Continuous mic -> event class | False positives per hour; on-device, since a mic in the home is a privacy surface |
| Machine condition monitoring | Manufacturing, predictive maintenance | Machine audio -> normal vs anomalous | Anomaly detection under domain shift; runs on edge hardware in a noisy plant |
| Bioacoustics and biodiversity | Conservation research (BirdNET) | Field-recorder audio -> species ID | Recall on rare classes; battery life on a solar-powered recorder |
| Emotion, language and intent triage | Contact-centre analytics | Call audio -> emotion, language, escalation flag | Consistency across accents; drives routing decisions |
| Cough and respiratory screening | Health | Cough audio -> condition score | Clinical validation and severe class imbalance |

**What the AudioSet mAP hides.** In an always-on system the metric that decides
whether you can ship is **false alarms per hour**, not accuracy - a smoke-alarm
detector that is 99% accurate but fires spuriously twice a day gets switched off
by the user, and a wake word that self-triggers once an hour is unusable no matter
what its top-1 says. **Domain shift** is the second killer: change the microphone,
the room, or the distance to the source, and a model tuned on clean benchmark clips
degrades hard, so a real evaluation set has to come from the deployment hardware.
Third, the world is **open-set** - production audio constantly contains sounds
outside the taxonomy, so a fixed-label classifier will confidently assign one of
its known classes to something it has never heard, which is why rejection
thresholds or a zero-shot model like CLAP matter more than another point of mAP.
Finally, benchmark labels themselves are noisy and wildly imbalanced, so a headline
mAP is an average over classes you may not care about at all.

---

## 3. How Modern Audio Classification Works

1. **Hand-crafted features + GMM/SVM (pre-2017).** MFCCs into a shallow classifier. Legacy.
2. **CNN on log-mel (VGGish, PANNs, 2019).** Treat the spectrogram as an image; large-scale AudioSet pretraining transfers well.
3. **Audio Spectrogram Transformer (AST, 2021).** A ViT over spectrogram patches - AudioSet state of the art and the default tagging backbone.
4. **Self-supervised encoders (wav2vec 2.0, HuBERT, WavLM, 2021).** Pretrain on unlabeled speech, fine-tune a light head for KWS, emotion, speaker ID; **BEATs** (2022) leads general audio tagging.
5. **CLAP (2023).** Contrastive language-audio pretraining enables **zero-shot** classification: score the clip against arbitrary text label prompts, no fine-tuning. By 2025-2026 CLAP-style zero-shot and unified audio encoders are the flexible default.

---

## 4. Evaluation Metrics

- **Accuracy / F1.** For single-label tasks (KWS, emotion, ESC-50).
- **mAP (mean Average Precision).** The standard for **multi-label** tagging (AudioSet) - averages precision across recall for each class.
- **d-prime / AUC.** Alternative multi-label summaries.
- **Confusion matrix.** Where the errors go - shown as an ECharts heatmap in the benchmark.

The cell computes accuracy and macro-F1 with `scikit-learn` on toy predictions.

---

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

y_true = ["dog", "rain", "dog", "engine", "rain"]
y_pred = ["dog", "rain", "cat", "engine", "rain"]
print("accuracy:", accuracy_score(y_true, y_pred))
print("macro-F1:", f1_score(y_true, y_pred, average="macro"))

accuracy: 0.8
macro-F1: 0.6666666666666666


## 5. The Model Landscape (mid-2026)

| Model | Params | License | Task | Best for |
|-------|--------|---------|------|----------|
| MIT/ast-finetuned-audioset-10-10-0.4593 | 86M | BSD-3 | AudioSet tagging (527) | general sound tagging |
| superb/hubert-large-superb-er | 300M | Apache 2.0 | speech emotion | emotion recognition |
| superb/wav2vec2-base-superb-ks | 95M | Apache 2.0 | keyword spotting (35) | command words |
| laion/clap-htsat-unfused | 150M | Apache 2.0 | zero-shot | label from text prompts |
| MIT/ast-finetuned-speech-commands-v2 | 86M | BSD-3 | keyword spotting | speech commands |

Benchmarks: **SUPERB** (speech tasks), **AudioSet mAP** (tagging), **HEAR** (holistic). All of these load through the `transformers` `audio-classification` (or `zero-shot-audio-classification`) pipeline.

---

## 6. Setup

Package roles:

- `transformers` (>=5.13) + `torch` - AST, HuBERT, wav2vec2, CLAP
- `datasets` - ESC-50 environmental-sound clips with labels
- `scikit-learn` - accuracy / F1 / confusion matrix
- `pyecharts` - accuracy bar + confusion-matrix heatmap

---

In [ ]:
import ctypes
import ctypes.util
import gc
import time
import urllib.request
from pathlib import Path

import torch
from dotenv import find_dotenv, load_dotenv

# Knowledge/.env sets HF_TOKEN - authenticated HF Hub requests get higher rate limits
load_dotenv(find_dotenv(usecwd=True))

device = "cuda:0" if torch.cuda.is_available() else "cpu"
if device != "cpu":
    print(torch.cuda.get_device_name(0))
print("device:", device)


def vram(tag=""):
    "Report current GPU memory (allocated / reserved). No-op on CPU."
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"VRAM {tag:16s} {alloc:5.2f} GB allocated / {reserved:5.2f} GB reserved")


def free_memory():
    "GC then release cached CPU/GPU memory. Call right after `del model`.\n\n    `del` drops the Python reference; this reclaims the RAM and hands the\n    freed VRAM back to the CUDA allocator so usage stays flat across cells.\n    "
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    # glibc keeps freed CPU allocations in its arenas instead of returning them
    # to the OS, so RSS compounds across model sections (cpu-offloaded weights
    # live in system RAM). malloc_trim(0) hands the freed arenas back. See
    # dl-visualization-and-memory.instructions.md - not optional on a 12 GB box.
    try:
        ctypes.CDLL(ctypes.util.find_library("c") or "libc.so.6").malloc_trim(0)
    except Exception:
        pass


# All downloads (samples, HF cache) go to DL_tasks/datasets/ (gitignored)
DATA_DIR = Path("../../datasets")
DATA_DIR.mkdir(exist_ok=True)

import numpy as np
from datasets import load_dataset

# ESC-50: 2000 labelled 5 s environmental-sound clips, 50 categories
esc = load_dataset("ashraq/esc50", split="train", cache_dir=str(DATA_DIR / "hf_cache"))
CATEGORIES = sorted(set(esc["category"]))
print(len(esc), "clips,", len(CATEGORIES), "categories")


def esc_clip(row):
    "Return {array, sampling_rate} for a transformers audio pipeline."
    a = row["audio"]
    return {"array": np.asarray(a["array"], dtype="float32"), "sampling_rate": a["sampling_rate"]}

NVIDIA GeForce RTX 3060
device: cuda:0


Repo card metadata block was not found. Setting CardData to empty.


2000 clips, 50 categories


## 7. AST - AudioSet tagging

The Audio Spectrogram Transformer tags a clip against AudioSet's 527 classes. The `audio-classification` pipeline handles resampling to 16 kHz and returns scored labels; `top_k` sets how many.

---

In [ ]:
from transformers import pipeline

tagger = pipeline("audio-classification", model="MIT/ast-finetuned-audioset-10-10-0.4593",
                  device=device, top_k=5)
clip = esc_clip(esc[0])
t0 = time.perf_counter()
print(f"true category: {esc[0]['category']}")
for p in tagger(clip):
    print(f"  {p['score']:.3f}  {p['label']}")
print(f"{time.perf_counter() - t0:.2f}s")

del tagger
free_memory()
vram("after ast")

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

true category: dog
  0.187  Bark
  0.185  Animal
  0.101  Dog
  0.084  Bow-wow
  0.071  Domestic animals, pets
  0.064  Canidae, dogs, wolves
  0.024  Sound effect
  0.015  Crow
  0.014  Caw
  0.014  Sigh
  0.014  Yip
  0.014  Growling
  0.013  Speech
  0.011  Music
  0.010  Silence
  0.009  Quack
  0.008  Gasp
  0.007  Hiccup
  0.007  Whimper (dog)
  0.007  Grunt
  0.007  Frog
  0.006  Plop
  0.005  Male speech, man speaking
  0.005  Whoosh, swoosh, swish
  0.004  Groan
  0.004  Inside, small room
  0.003  Roar
  0.003  Duck
  0.002  Cattle, bovinae
  0.002  Roaring cats (lions, tigers)
  0.002  Sneeze
  0.002  Wild animals
  0.002  Livestock, farm animals, working animals
  0.002  Cat
  0.002  Screaming
  0.002  Zipper (clothing)
  0.002  Slap, smack
  0.002  Bird
  0.002  Breathing
  0.002  Bang
  0.002  Musical instrument
  0.001  Outside, rural or natural
  0.001  Boing
  0.001  Fart
  0.001  Whistle
  0.001  Whack, thwack
  0.001  Pig
  0.001  Ding-dong
  0.001  Ringtone
  0.001 

## 8. CLAP - zero-shot classification

CLAP scores the clip against **arbitrary text prompts**, so it classifies ESC-50 with no fine-tuning - we just pass the 50 category names as candidate labels. This is the flexible modern default.

---

In [ ]:
zsc = pipeline("zero-shot-audio-classification", model="laion/clap-htsat-unfused", device=device)
prompts = [f"the sound of {c.replace('_', ' ')}" for c in CATEGORIES]

clip = esc_clip(esc[0])
t0 = time.perf_counter()
preds = zsc(clip["array"], candidate_labels=prompts)
print(f"true: {esc[0]['category']}  ->  top: {preds[0]['label']}  ({preds[0]['score']:.3f})")
print(f"{time.perf_counter() - t0:.2f}s")

del zsc
free_memory()
vram("after clap")

Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

true: dog  ->  top: the sound of dog  (0.735)
0.34s
VRAM after clap        0.01 GB allocated /  0.02 GB reserved


## 9. Head-to-head Benchmark

Compare **CLAP zero-shot** against a **fused CLAP** variant on an ESC-50 subset: **accuracy** + **RTF**, plus a confusion-matrix heatmap for the zero-shot model. Same clips, same label set. A 50-clip subset is a smoke test; ESC-50's official 5-fold protocol is the real evaluation.

---

In [ ]:
# ECharts (pyecharts) is the repo standard for all charts - it renders interactive
# and embeds straight into the Quarto docs via .render_notebook().
from pyecharts import options as opts
from pyecharts.charts import Bar


def bar_chart(title, categories, series, y_name=""):
    "Grouped bar chart. `series` is a dict {name: [values aligned to categories]}."
    chart = Bar(init_opts=opts.InitOpts(width="720px", height="420px"))
    chart.add_xaxis([str(c) for c in categories])
    for name, vals in series.items():
        chart.add_yaxis(name, [round(float(v), 4) for v in vals])
    chart.set_global_opts(
        title_opts=opts.TitleOpts(title=title),
        yaxis_opts=opts.AxisOpts(name=y_name),
        xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=20)),
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
        legend_opts=opts.LegendOpts(pos_top="8%"),
    )
    return chart.render_notebook()

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

N = 50  # clips to evaluate
subset = esc.select(range(N))
refs = [row["category"] for row in subset]
prompts = [f"the sound of {c.replace('_', ' ')}" for c in CATEGORIES]

results, preds_for_cm = {}, None
for model_id in ["laion/clap-htsat-unfused", "laion/clap-htsat-fused"]:
    zsc = pipeline("zero-shot-audio-classification", model=model_id, device=device)
    t0 = time.perf_counter()
    hyps = []
    for row in subset:
        p = zsc(esc_clip(row)["array"], candidate_labels=prompts)
        top = p[0]["label"]
        hyps.append(CATEGORIES[prompts.index(top)])
    elapsed = time.perf_counter() - t0
    acc = accuracy_score(refs, hyps)
    results[model_id.split("/")[-1]] = {"acc": acc, "rtf": elapsed / (N * 5.0)}
    print(f"{model_id:30s} acc {acc:6.2%}  {elapsed:5.1f}s")
    if preds_for_cm is None:
        preds_for_cm = hyps
    del zsc
    free_memory()
vram("after benchmark")

Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


laion/clap-htsat-unfused       acc 100.00%    3.2s


Loading weights:   0%|          | 0/477 [00:00<?, ?it/s]

laion/clap-htsat-fused         acc 96.00%    3.4s
VRAM after benchmark   0.01 GB allocated /  0.02 GB reserved


In [ ]:
names = list(results)
bar_chart("Audio classification: zero-shot accuracy on ESC-50 subset",
          names, {"accuracy": [results[n]["acc"] for n in names]}, y_name="accuracy")

In [ ]:
# Confusion matrix as an ECharts heatmap (categories that actually appear in the subset)
from pyecharts import options as opts
from pyecharts.charts import HeatMap

present = sorted(set(refs))
cm = confusion_matrix(refs, preds_for_cm, labels=present)
data = [[j, i, int(cm[i][j])] for i in range(len(present)) for j in range(len(present))]

hm = HeatMap(init_opts=opts.InitOpts(width="760px", height="620px"))
hm.add_xaxis(present)
hm.add_yaxis("true vs predicted", present, data,
             label_opts=opts.LabelOpts(is_show=True, position="inside"))
hm.set_global_opts(
    title_opts=opts.TitleOpts(title="ESC-50 confusion (CLAP unfused)"),
    xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=45)),
    visualmap_opts=opts.VisualMapOpts(max_=int(cm.max()), orient="vertical", pos_left="right"),
)
hm.render_notebook()

## 10. Live Microphone Tagging

The same two models from sections 7 and 8, pointed at the machine's actual microphone instead of an ESC-50 clip. Recording a few seconds of whatever is around you is the fastest way to feel the difference between the two designs: **AST** can only answer in AudioSet's fixed 527 classes, while **CLAP** scores the same audio against a label set you type in the cell and can change between two consecutive calls.

Mind the sample rates - the two pipelines do not behave the same way:

- `audio-classification` (AST) accepts a `{"array", "sampling_rate"}` dict and **resamples for you**, so a dict is always safe.
- `zero-shot-audio-classification` (CLAP) rejects dicts and takes a bare array that it assumes is **already at the model's 48 kHz**. Hand it 16 kHz audio and it will not complain, it will just score a signal playing at a third of its true speed.

So this cell records once, keeps a 48 kHz copy for CLAP, and derives the 16 kHz copy for AST.

Needs a working capture device at `/dev/snd`; the cell raises rather than skipping,
so a muted or missing mic is visible instead of looking like a clean run. The docs
builder never executes it (`skip_exec: true`).

---

In [ ]:
# --- standalone setup ----------------------------------------------------------
# Lifted from the Setup section and the helper cells above so this demo runs on its
# own in a fresh kernel - no earlier cell has to have been executed first.

import ctypes
import ctypes.util
import gc
import numpy as np
import torch

from dotenv import find_dotenv, load_dotenv
from pathlib import Path
from transformers import pipeline

# Knowledge/.env sets HF_TOKEN - authenticated HF Hub requests get higher rate limits
load_dotenv(find_dotenv(usecwd=True))

device = "cuda:0" if torch.cuda.is_available() else "cpu"

def vram(tag=""):
    "Report current GPU memory (allocated / reserved). No-op on CPU."
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"VRAM {tag:16s} {alloc:5.2f} GB allocated / {reserved:5.2f} GB reserved")

def free_memory():
    "GC then release cached CPU/GPU memory. Call right after `del model`.\n\n    `del` drops the Python reference; this reclaims the RAM and hands the\n    freed VRAM back to the CUDA allocator so usage stays flat across cells.\n    "
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    # glibc keeps freed CPU allocations in its arenas instead of returning them
    # to the OS, so RSS compounds across model sections (cpu-offloaded weights
    # live in system RAM). malloc_trim(0) hands the freed arenas back. See
    # dl-visualization-and-memory.instructions.md - not optional on a 12 GB box.
    try:
        ctypes.CDLL(ctypes.util.find_library("c") or "libc.so.6").malloc_trim(0)
    except Exception:
        pass

# All downloads (samples, HF cache) go to DL_tasks/datasets/ (gitignored)
DATA_DIR = Path("../../datasets")

DATA_DIR.mkdir(exist_ok=True)

# --- the demo ------------------------------------------------------------------
# sounddevice needs the PortAudio runtime (libportaudio2) and ALSA nodes under
# /dev/snd. On the knowledge-lab LXC those are passed in from the Proxmox host by
# `av_devices` in infra/proxmox/variables.tf - the camera's video node and its
# USB-Audio card are separate passthroughs.
import librosa
import sounddevice as sd
from IPython.display import Audio as AudioPlayer  # `Audio` may already be datasets.Audio

# Substring of the *PortAudio* device name, which is not the ALSA card id: this mic
# is card id "U2K" but shows up as "UGREEN camera 2K: USB Audio (hw:0,0)". Run
# `python -c "import sounddevice; print(sounddevice.query_devices())"` to list them.
MIC_HINT = "UGREEN"  # None -> just take the first input device


def pick_microphone(hint=MIC_HINT):
    "Return (device_index, native_sample_rate) for a capture device, preferring `hint`."
    inputs = [(i, d) for i, d in enumerate(sd.query_devices()) if d["max_input_channels"] > 0]
    if not inputs:
        raise RuntimeError(
            "no audio capture device: PortAudio sees no ALSA card. Check that /dev/snd "
            "exists and holds controlC*/pcmC*D*c nodes - on the LXC that means adding "
            "them to av_devices and running `just tf-apply`."
        )
    match = [(i, d) for i, d in inputs if hint and hint.lower() in d["name"].lower()]
    idx, info = (match or inputs)[0]
    return idx, int(info["default_samplerate"])


def record(seconds, target_sr):
    "Record mono from the mic, resample to target_sr, report the level, return float32."
    mic, native_sr = pick_microphone()
    print(f"mic [{mic}] @ {native_sr} Hz - recording {seconds}s...")
    rec = sd.rec(int(seconds * native_sr), samplerate=native_sr, channels=1,
                 dtype="float32", device=mic)
    sd.wait()
    audio = rec.squeeze()

    # Record at the rate the hardware actually offers, then resample - not the
    # reverse. Asking ALSA for a rate the device does not have is the usual cause
    # of a result that sounds (and transcribes) sped-up or slowed-down.
    if native_sr != target_sr:
        audio = librosa.resample(audio, orig_sr=native_sr, target_sr=target_sr)

    peak = float(np.abs(audio).max())
    dbfs = 20 * np.log10(peak) if peak > 0 else float("-inf")
    print(f"  {audio.size / target_sr:.1f}s captured, peak {dbfs:.1f} dBFS")
    if peak < 1e-4:
        # Every model here will happily invent an answer from digital silence.
        raise RuntimeError("captured silence - the mic is muted (Mic Capture Switch is off)")
    if peak > 1.0:
        # PortAudio hands back float32 that can overshoot full scale when the
        # capture gain is too high. Past 1.0 the take is already distorted; all we
        # can do is scale it back into range and say so.
        print(f"  overloaded ({dbfs:+.1f} dBFS, clipped) - lower 'Mic Capture Volume'")
        audio = audio / peak
    elif peak < 0.05:
        print(f"  quiet take, scaling {0.5 / peak:.0f}x "
              "(raise 'Mic Capture Volume' with alsamixer -c 0 to fix it properly)")
        audio = audio / peak * 0.5
    return audio


SECONDS = 5
CLAP_SR, AST_SR = 48_000, 16_000  # each model's native feature-extractor rate

live_48k = record(SECONDS, CLAP_SR)
live_16k = librosa.resample(live_48k, orig_sr=CLAP_SR, target_sr=AST_SR)
display(AudioPlayer(live_48k, rate=CLAP_SR))

# --- AST: fixed AudioSet taxonomy. Dict input, so the pipeline handles the rate.
tagger = pipeline("audio-classification", model="MIT/ast-finetuned-audioset-10-10-0.4593",
                  device=device, top_k=5)
print("AST (AudioSet, 527 fixed classes):")
for p in tagger({"array": live_16k, "sampling_rate": AST_SR}):
    print(f"  {p['score']:.3f}  {p['label']}")

del tagger  # one large model live at a time
free_memory()

# --- CLAP: same audio, but you choose the classes. Edit MY_SOUNDS and re-run.
MY_SOUNDS = ["speech", "typing on a keyboard", "a fan or air conditioner",
             "music", "silence", "a door closing"]

zsc = pipeline("zero-shot-audio-classification", model="laion/clap-htsat-unfused", device=device)
print("\nCLAP (zero-shot, your own labels):")
for p in zsc(live_48k, candidate_labels=[f"the sound of {s}" for s in MY_SOUNDS]):
    print(f"  {p['score']:.3f}  {p['label']}")

del zsc
free_memory()
vram("after live mic")

## 11. Going Further

- **Multi-label tagging at scale.** Evaluate AST with **mAP** on the full AudioSet eval set; **BEATs** and CED push the state of the art.
- **Fine-tuning.** Fine-tune wav2vec2 / HuBERT for your own KWS, emotion, or speaker-ID head: [HF audio classification guide](https://huggingface.co/docs/transformers/tasks/audio_classification).
- **Speaker ID / diarization.** `speechbrain`'s ECAPA-TDNN embeddings for verification and clustering.
- **Zero-shot everywhere.** Swap the CLAP candidate prompts to classify any new taxonomy with no training data.
- **Always-on tagging.** Loop section 10 over a rolling buffer to build a continuous sound monitor - the shape of every "what is that noise" appliance.

---